In [ ]:
# import json, subprocess, sys, tempfile, os

# def run_and_check(script_path: str, contract_schema: dict, timeout: int = 120):
#     """执行脚本，要求其向 stdout 打印一行 JSON 结果，并校验关键字段。"""
#     proc = subprocess.run(
#         [sys.executable, script_path],
#         capture_output=True, text=True, timeout=timeout,
#         cwd=os.path.dirname(script_path) or ".",
#     )
#     if proc.returncode != 0:
#         # 语法/运行时错误：把 traceback 完整回灌给模型，让它自己修
#         return {"ok": False, "stage": "exec", "stderr": proc.stderr[-4000:]}

#     try:
#         result = json.loads(proc.stdout.strip().splitlines()[-1])
#     except (json.JSONDecodeError, IndexError):
#         return {"ok": False, "stage": "contract", "stderr": proc.stdout[-2000:]}

#     problems = []
#     for key, rule in contract_schema.items():
#         if key not in result:
#             problems.append(f"missing:{key}")
#             continue
#         lo, hi = rule.get("range", (float("-inf"), float("inf")))
#         if not (lo <= result[key] <= hi):
#             problems.append(f"out_of_range:{key}={result[key]}")
#     if problems:
#         return {"ok": False, "stage": "assert", "problems": problems}
#     return {"ok": True, "result": result}

# # 例：要求脚本报告样本量、估计值、标准误、95% 置信区间
# schema = {
#     "n":        {"range": (1, 10**7)},
#     "estimate": {"range": (-1e6, 1e6)},
#     "se":       {"range": (0, 1e6)},
#     "ci_low":   {"range": (-1e6, 1e6)},
#     "ci_high":  {"range": (-1e6, 1e6)},
# }
# print(run_and_check("analysis.py", schema))


In [1]:
import numpy as np

def bootstrap_ci_reference(x, B=2000, alpha=0.05, seed=0):
    """正确实现：有放回重抽，取均值抽样分布的百分位数。"""
    rng = np.random.default_rng(seed)
    n = len(x)
    idx = rng.integers(0, n, size=(B, n))
    stat = x[idx].mean(axis=1)
    lo, hi = np.percentile(stat, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return lo, hi

def bootstrap_ci_ai(x, B=2000, alpha=0.05, seed=0):
    """AI 生成版本：结构与正确实现相同，但重抽时不慎写成了无放回。"""
    rng = np.random.default_rng(seed)
    n = len(x)
    idx = np.array([rng.choice(n, size=n, replace=False) for _ in range(B)])  # bug
    stat = x[idx].mean(axis=1)
    lo, hi = np.percentile(stat, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return lo, hi

master = np.random.default_rng(20260101)   # 测试自身的种子也要固定

def prop_shift(fn):
    """平移等变性：数据加常数 c，区间两端应同样平移 c。"""
    x = master.normal(0, 1, 60)
    lo1, hi1 = fn(x, seed=1)
    lo2, hi2 = fn(x + 100.0, seed=1)
    return bool(np.allclose([lo1 + 100, hi1 + 100], [lo2, hi2], atol=1e-8))

def prop_nondegenerate(fn, reps=20):
    """非退化性：对有方差的样本，区间宽度必须为正且量级合理。"""
    for _ in range(reps):
        x = master.normal(0, 1, 40)
        lo, hi = fn(x, seed=2)
        if not (hi - lo) > 0.1:   # n=40 的正态均值 95% 区间宽度约 0.6
            return False
    return True

def prop_coverage(fn, reps=400, n=50, alpha=0.05, mu=0.5):
    """覆盖率：在真值已知的模拟数据上，覆盖率应接近名义水平 95%。"""
    cover = 0
    for r in range(reps):
        x = master.normal(mu, 1.0, n)
        lo, hi = fn(x, seed=1000 + r)
        cover += (lo <= mu <= hi)
    return cover / reps

for name, fn in [("reference", bootstrap_ci_reference), ("ai_version", bootstrap_ci_ai)]:
    print(f"{name:10s} shift={prop_shift(fn)}  "
          f"nondegenerate={prop_nondegenerate(fn)}  "
          f"coverage={prop_coverage(fn):.3f}")

reference  shift=True  nondegenerate=True  coverage=0.938
ai_version shift=True  nondegenerate=False  coverage=0.000


In [2]:
import ast

RANDOM_FUNCS = {"random", "choice", "integers", "normal", "uniform", "permutation",
                "shuffle", "sample", "rand", "randn", "randint", "beta", "binomial", "poisson"}
SEED_FUNCS = {"seed", "default_rng", "Generator", "set_state"}

def call_name(node):
    f = node.func
    if isinstance(f, ast.Attribute):
        return f.attr
    if isinstance(f, ast.Name):
        return f.id
    return ""

def check_source(src):
    tree = ast.parse(src)
    problems = []
    # 规则 A：模块顶层除 import、def、class、常量赋值、__main__ 守卫外都标记
    for node in tree.body:
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef,
                             ast.Import, ast.ImportFrom)):
            continue
        if isinstance(node, ast.Assign) and isinstance(node.value, ast.Constant):
            continue
        if isinstance(node, ast.If) and "main" in ast.dump(node.test):
            continue
        kind = "顶层可变赋值" if isinstance(node, ast.Assign) else "顶层执行语句"
        problems.append(f"global_state: 第 {node.lineno} 行附近存在{kind}（{type(node).__name__}）")
    # 规则 B：随机性调用与种子设置
    calls = [n for n in ast.walk(tree) if isinstance(n, ast.Call)]
    has_seed = any(call_name(c) in SEED_FUNCS for c in calls)
    uses_random = any(call_name(c) in RANDOM_FUNCS for c in calls)
    if uses_random and not has_seed:
        problems.append("no_seed: 检测到随机性调用，但未见 seed / default_rng")
    return problems

BAD = '''
import numpy as np
import pandas as pd

df = pd.read_csv("data.csv")
df["z"] = np.random.normal(0, 1, len(df))
result = df.groupby("group")["z"].mean()
print(result)
'''

GOOD = '''
import numpy as np
import pandas as pd

SEED = 20260101

def add_noise(df, seed=SEED):
    rng = np.random.default_rng(seed)
    out = df.copy()
    out["z"] = rng.normal(0, 1, len(out))
    return out

def main():
    df = pd.read_csv("data.csv")
    print(add_noise(df).groupby("group")["z"].mean())

if __name__ == "__main__":
    main()
'''

for label, src in [("BAD（AI 生成，含两类问题）", BAD), ("GOOD（修正后）", GOOD)]:
    print(label)
    if check_source(src):
        for p in check_source(src):
            print("  -", p)
    else:
        print("  - 通过")

BAD（AI 生成，含两类问题）
  - global_state: 第 5 行附近存在顶层可变赋值（Assign）
  - global_state: 第 6 行附近存在顶层可变赋值（Assign）
  - global_state: 第 7 行附近存在顶层可变赋值（Assign）
  - global_state: 第 8 行附近存在顶层执行语句（Expr）
  - no_seed: 检测到随机性调用，但未见 seed / default_rng
GOOD（修正后）
  - 通过


In [ ]:
# import numpy as np, pandas as pd, json
# SEED = 20260101
# np.random.seed(SEED)

# # ---- 阶段 1：读取 ----
# df = pd.read_csv("data/patient_data.csv")
# assert df.shape[0] > 0, "空数据"
# print(df.dtypes.to_string())
# print("缺失数：\n" + df.isna().sum().to_string())

# # ---- 阶段 2：清洗（策略须经确认后写入） ----
# num_cols = df.select_dtypes("number").columns
# miss = df[num_cols].isna().mean()
# print("缺失比例：\n" + miss[miss > 0].to_string())
# # 例：中位数插补（记录被插补的列与中位数）
# fill_values = df[num_cols].median()
# df[num_cols] = df[num_cols].fillna(fill_values)

# # ---- 阶段 3：探索 ----
# desc = df.groupby("treatment_group")[num_cols].agg(["mean", "std", "median"])
# print(desc.round(3).to_string())

# # ---- 阶段 4：建模 ----
# import statsmodels.formula.api as smf
# fit = smf.ols("sbp_followup ~ treatment_group + age + bmi + sbp_baseline", data=df).fit()
# print(fit.summary().tables[1])

# # ---- 阶段 5：诊断 ----
# infl = fit.get_influence()
# cooks = infl.cooks_distance[0]
# print("Cook 距离超过 4/n 的观测数：", int((cooks > 4 / len(df)).sum()))
# vif = pd.Series(
#     [1 / (1 - smf.ols(f"{c} ~ " + " + ".join([x for x in ["age", "bmi", "sbp_baseline"] if x != c]),
#                       data=df).fit().rsquared)
#      for c in ["age", "bmi", "sbp_baseline"]],
#     index=["age", "bmi", "sbp_baseline"], name="VIF")
# print(vif.round(2).to_string())

# # ---- 阶段 6：输出结构化结果 ----
# key = "treatment_group[T.drug]"
# res = {
#     "n": int(fit.nobs), "estimate": float(fit.params[key]), "se": float(fit.bse[key]),
#     "ci_low": float(fit.conf_int().loc[key, 0]), "ci_high": float(fit.conf_int().loc[key, 1]),
#     "method": "OLS with age/bmi/baseline adjustment", "seed": SEED,
# }
# print(json.dumps(res))   # 供 run_check.py 消费

In [ ]:
# import pandas as pd
# df = pd.read_csv('patient_data.csv')
# print(df.info()); print(df.isnull().sum()); print(df.describe())

In [ ]:
# from scipy import stats
# stat_sw, p_sw = stats.shapiro(df['sbp_followup'])
# if p_sw > 0.05:
#     t_stat, p_val = stats.ttest_ind(
#         df[df['treatment_group']=='drug']['sbp_followup'],
#         df[df['treatment_group']=='placebo']['sbp_followup'])
# else:
#     u_stat, p_val = stats.mannwhitneyu(
#         df[df['treatment_group']=='drug']['sbp_followup'],
#         df[df['treatment_group']=='placebo']['sbp_followup'])

In [ ]:
# import numpy as np
# grp1 = df[df['treatment_group']=='drug']['sbp_followup']
# grp2 = df[df['treatment_group']=='placebo']['sbp_followup']
# n1, n2 = len(grp1), len(grp2)
# sd1, sd2 = grp1.std(ddof=1), grp2.std(ddof=1)
# pooled_sd = np.sqrt(((n1 - 1) * sd1**2 + (n2 - 1) * sd2**2) / (n1 + n2 - 2))
# cohens_d = (grp1.mean() - grp2.mean()) / pooled_sd
# se_d = np.sqrt((n1 + n2) / (n1 * n2) + cohens_d**2 / (2 * (n1 + n2)))
# ci_lower, ci_upper = cohens_d - 1.96 * se_d, cohens_d + 1.96 * se_d

In [4]:
# import numpy as np, json, re
# from pathlib import Path

# def chunk(text, size=800, overlap=150):
#     """按字符数切分并保留重叠，返回 (块, 起始位置)。"""
#     step = size - overlap
#     return [(text[i:i + size], i) for i in range(0, len(text), step)]

# def embed(chunks, model):
#     # model 为任一嵌入模型封装，返回 shape=(n, d) 的二维数组
#     return np.asarray(model.encode(chunks), dtype="float32")

# def retrieve(query_vec, doc_vecs, k=5):
#     # 余弦相似度；向量已归一化时点积即余弦
#     sims = doc_vecs @ query_vec / (np.linalg.norm(doc_vecs, axis=1) * np.linalg.norm(query_vec) + 1e-12)
#     return np.argsort(-sims)[:k], np.sort(sims)[::-1][:k]

# # 检索结果必须携带 (doc_id, page, char_start)，否则无法溯源

In [ ]:
# import matplotlib as mpl, matplotlib.pyplot as plt

# mpl.rcParams.update({
#     "figure.dpi": 150,              # 屏幕预览
#     "savefig.dpi": 600,             # 存盘分辨率
#     "savefig.bbox": "tight",
#     "savefig.format": "pdf",
#     "pdf.fonttype": 42,             # TrueType 嵌入，避免字体替换
#     "ps.fonttype": 42,
#     "font.family": "serif",
#     "font.serif": ["Times New Roman", "DejaVu Serif"],
#     "mathtext.fontset": "stix",     # 公式字体与正文字体协调
#     "font.size": 9,
#     "axes.labelsize": 10,
#     "axes.titlesize": 10,
#     "xtick.labelsize": 8,
#     "ytick.labelsize": 8,
#     "legend.fontsize": 8,
#     "legend.frameon": False,
#     "axes.linewidth": 0.8,
#     "lines.linewidth": 1.2,
#     "xtick.direction": "in",
#     "ytick.direction": "in",
# })

# # 色盲友好调色板（Okabe-Ito 八色）
# OKABE_ITO = ["#000000", "#E69F00", "#56B4E9", "#009E73",
#              "#F0E442", "#0072B2", "#D55E00", "#CC79A7"]

# fig, ax = plt.subplots(figsize=(3.5, 2.8))     # 单栏宽度约 85 mm ≈ 3.35 in
# # ...
# fig.savefig("fig1.pdf")

In [5]:
import numpy as np
from scipy.stats import norm

def n_per_arm_two_prop(p1, p2, alpha=0.05, power=0.80):
    z_a = norm.ppf(1 - alpha / 2); z_b = norm.ppf(power)
    return (z_a + z_b)**2 * (p1*(1-p1) + p2*(1-p2)) / (p2 - p1)**2

print(round(n_per_arm_two_prop(0.70, 0.73)))            # 约 3550

def n_per_arm_mean(delta, sigma, alpha=0.05, power=0.80):
    z_a = norm.ppf(1 - alpha / 2); z_b = norm.ppf(power)
    return (z_a + z_b)**2 * 2 * sigma**2 / delta**2

print(round(n_per_arm_mean(delta=0.5, sigma=2.0)))      # 连续响应示例

3550
251


In [ ]:
import itertools, numpy as np, pandas as pd
import statsmodels.formula.api as smf

rng = np.random.default_rng(20260101)

# 1. 因子与水平：模型、检索 k、是否启用思维链
factors = {"model": ["small", "large"], "top_k": [3, 10], "cot": [0, 1]}
design = pd.DataFrame(list(itertools.product(*factors.values())), columns=factors.keys())

# 2. 随机化运行顺序 + 日期区组（每天最多跑 4 个配置）
design = design.sample(frac=1, random_state=20260101).reset_index(drop=True)
design["block"] = design.index // 4

# 3. 执行每个配置并收集响应（此处用占位函数代表真实运行）
def run_config(row):
    # 真实场景中：调用模型 API，返回准确率与成本
    return {"accuracy": 0.70 + 0.05*row["cot"] + 0.02*row["top_k"]/10
                        + 0.06*(row["model"] == "large") + rng.normal(0, 0.01),
            "cost": 1.0 + 4.0*(row["model"] == "large") + 0.1*row["top_k"]}
design = pd.concat([design, design.apply(run_config, axis=1).apply(pd.Series)], axis=1)

# 4. 拟合含二阶交互的模型，区组作为固定效应吸收批次差异
mdl = smf.ols("accuracy ~ C(model)*C(top_k)*C(cot) + C(block)", data=design).fit()
print(mdl.summary().tables[1])

# 5. 主效应与交互效应的效应量（以标准差为单位）
effects = mdl.params.drop(labels=[p for p in mdl.params.index if "block" in p])
print((effects / design["accuracy"].std()).round(3).sort_values(key=abs, ascending=False))

In [ ]:
# import hashlib, json
# def sha256(path, chunk=1 << 20):
#     h = hashlib.sha256()
#     with open(path, "rb") as f:
#         for b in iter(lambda: f.read(chunk), b""):
#             h.update(b)
#     return h.hexdigest()

# manifest = {"data": {"patient_data.csv": sha256("data/patient_data.csv")},
#             "seed": 20260101, "model": "claude-sonnet-4-5@2026-01-01"}
# json.dump(manifest, open("run_manifest.json", "w"), indent=2)

In [ ]:
# import mlflow
# with mlflow.start_run(run_name="ols_adjusted"):
#     mlflow.log_params({"seed": 20260101, "model": "ols", "covariates": "age+bmi+baseline"})
#     mlflow.log_metrics({"estimate": res["estimate"], "se": res["se"]})
#     mlflow.log_artifact("fig1.pdf")
#     mlflow.log_artifact("run_manifest.json")

In [8]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

rng = np.random.default_rng(20260101)
G, m = 40, 25                       # 40 个聚类，每个聚类 25 个观测，n = 1000
cid = np.repeat(np.arange(G), m)
u = rng.normal(0, 1.0, G)[cid]      # 聚类层面随机效应：组内相关的来源
x = rng.normal(0, 1, G)[cid]        # 协变量在聚类层面取值（如 cluster 级干预）
y = 1.0 + 0.3 * x + u + rng.normal(0, 1, G * m)
dat = pd.DataFrame({"y": y, "x": x, "cid": cid})

fit0 = smf.ols("y ~ x", dat).fit()
fit1 = smf.ols("y ~ x", dat).fit(cov_type="HC1")
fitc = smf.ols("y ~ x", dat).fit(cov_type="cluster", cov_kwds={"groups": dat["cid"]})

print("coef(x)       =", round(fit0.params["x"], 4))
print("SE 经典       =", round(fit0.bse["x"], 4))
print("SE HC1        =", round(fit1.bse["x"], 4))
print("SE cluster    =", round(fitc.bse["x"], 4))
print("SE比值 cluster/经典 =", round(fitc.bse["x"] / fit0.bse["x"], 2))
print("p 经典        =", fit0.pvalues["x"])
print("p 聚类        =", fitc.pvalues["x"])

# 系数第二条路径核对：正规方程
Xm = np.c_[np.ones(len(dat)), dat["x"].to_numpy()]
beta = np.linalg.lstsq(Xm, dat["y"].to_numpy(), rcond=None)[0]
print("coef lstsq    =", round(beta[1], 4))

# 组内相关诊断：单因素随机效应模型的方差分量矩估计
gmean = dat.groupby("cid")["y"].mean()
grand = dat["y"].mean()
ssw = ((dat["y"] - dat["cid"].map(gmean)) ** 2).sum()
ssb = m * ((gmean - grand) ** 2).sum()
msw = ssw / (G * (m - 1))
msb = ssb / (G - 1)
icc = (msb - msw) / (msb + (m - 1) * msw)
print("ICC 矩估计    =", round(icc, 3))

dat.to_csv("cluster_data.csv", index=False)   # 供 R 侧互核使用

coef(x)       = 0.4384
SE 经典       = 0.0516
SE HC1        = 0.0466
SE cluster    = 0.1557
SE比值 cluster/经典 = 3.02
p 经典        = 6.962460328652604e-17
p 聚类        = 0.004885074724560043
coef lstsq    = 0.4384
ICC 矩估计    = 0.57
